# Importing Libraries

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import polars as pl
import numpy as np
from google.colab import drive
import time
from collections import defaultdict

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


# Defining File Paths

In [ ]:
drive_path = '/content/drive/Shareddrives/CMPE256_FinalProject/board_game_recommendation'
foundation_path = f"{drive_path}/processed_data/final_foundation"
split_dir = f"{foundation_path}/splits"
variant_v2_dir = f"{drive_path}/processed_data/variant_llm_v2"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Load Datasets

In [ ]:
train_df = pl.read_parquet(f"{split_dir}/train.parquet")
val_df = pl.read_parquet(f"{split_dir}/val.parquet")
test_df = pl.read_parquet(f"{split_dir}/test.parquet")

Loading split data...


In [ ]:
# Loading Item Metadata and LLM Embeddings
items_df = pl.read_parquet(f"{split_dir}/items_metadata_final.parquet")
emb_v2_df = pl.read_parquet(f"{variant_v2_dir}/item_embeddings_v2_768.parquet").sort("item_id")

In [ ]:
# Dynamic ID Counting; using +1 because IDs are 0-indexed
ACTUAL_NUM_USERS = int(train_df["user_id"].max() + 1)
ACTUAL_NUM_ITEMS = int(max(train_df["item_id"].max(), items_df["item_id"].max()) + 1)

In [ ]:
print(f"Users found in data: {ACTUAL_NUM_USERS:,}")
print(f"Items found in data: {ACTUAL_NUM_ITEMS:,}")

Users found in data: 411,375
Items found in data: 21,925


In [ ]:
raw_embeddings = np.stack(emb_v2_df["embedding_llm"].to_list())
if raw_embeddings.shape[0] < ACTUAL_NUM_ITEMS:
    padding = np.zeros((ACTUAL_NUM_ITEMS - raw_embeddings.shape[0], 768))
    raw_embeddings = np.vstack([raw_embeddings, padding])

item_embeddings_v2_tensor = torch.tensor(raw_embeddings, dtype=torch.float32)
print(f"Users: {ACTUAL_NUM_USERS:,} | Items: {ACTUAL_NUM_ITEMS:,}")
print(f"V2 Embedding Tensor Shape: {item_embeddings_v2_tensor.shape}")

Users: 411,375 | Items: 21,925
V2 Embedding Tensor Shape: torch.Size([21925, 768])


In [ ]:
class BoardGameDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df["user_id"].to_numpy(), dtype=torch.long)
        self.items = torch.tensor(df["item_id"].to_numpy(), dtype=torch.long)
        self.ratings = torch.tensor(df["Rating"].to_numpy(), dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.ratings[idx]

In [ ]:
BATCH_SIZE = 16384
train_loader = DataLoader(BoardGameDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(BoardGameDataset(val_df), batch_size=BATCH_SIZE)
test_loader = DataLoader(BoardGameDataset(test_df), batch_size=BATCH_SIZE)

Hybrid LLM Augmented Architecture

In [ ]:
class HybridLLMModelV2(nn.Module):
    def __init__(self, num_users, num_items, llm_embs, latent_dim=64):
        super().__init__()
        # Collaborative Towers
        self.user_emb = nn.Embedding(num_users, latent_dim)
        self.item_emb = nn.Embedding(num_items, latent_dim)

        #Frozen Semantic Tower
        self.register_buffer('llm_feat', llm_embs)

        #Hybrid MLP Head
        input_dim = latent_dim + latent_dim + 768
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, u_id, i_id):
        u_vec = self.user_emb(u_id)
        i_vec = self.item_emb(i_id)
        l_vec = self.llm_feat[i_id]
        combined = torch.cat([u_vec, i_vec, l_vec], dim=1)
        return self.mlp(combined).squeeze()

In [ ]:
model = HybridLLMModelV2(ACTUAL_NUM_USERS, ACTUAL_NUM_ITEMS, item_embeddings_v2_tensor).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

In [ ]:
def run_evaluation(model, loader):
    model.eval()
    mse = 0
    with torch.no_grad():
        for u, i, r in loader:
            u, i, r = u.to(device), i.to(device), r.to(device)
            preds = model(u, i)
            mse += nn.functional.mse_loss(preds, r, reduction='sum').item()
    return np.sqrt(mse / len(loader.dataset))

best_val_rmse = float('inf')
early_stop_patience = 3
epochs_no_improve = 0

# Training

In [ ]:
for epoch in range(1, 16):
    start_time = time.time()
    model.train()
    epoch_loss = 0
    for u, i, r in train_loader:
        u, i, r = u.to(device), i.to(device), r.to(device)
        optimizer.zero_grad()
        preds = model(u, i)
        loss = criterion(preds, r)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * u.size(0)

    train_rmse = np.sqrt(epoch_loss / len(train_loader.dataset))
    val_rmse = run_evaluation(model, val_loader)
    scheduler.step(val_rmse)

    print(f"Epoch {epoch:02d} | Train: {train_rmse:.4f} | Val: {val_rmse:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        epochs_no_improve = 0
        torch.save(model.state_dict(), f"{variant_v2_dir}/best_hybrid_v2_total_context.pth")
        print("  --> New Best V2 Model Saved!")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= early_stop_patience:
            print(f"Early stopping at epoch {epoch}")
            break

Epoch 01 | Train: 1.3588 | Val: 1.2520 | LR: 0.001000
  --> New Best V2 Model Saved!
Epoch 02 | Train: 1.2663 | Val: 1.2001 | LR: 0.001000
  --> New Best V2 Model Saved!
Epoch 03 | Train: 1.2278 | Val: 1.1865 | LR: 0.001000
  --> New Best V2 Model Saved!
Epoch 04 | Train: 1.2070 | Val: 1.1791 | LR: 0.001000
  --> New Best V2 Model Saved!
Epoch 05 | Train: 1.1929 | Val: 1.1804 | LR: 0.001000
Epoch 06 | Train: 1.1833 | Val: 1.1808 | LR: 0.000500
Epoch 07 | Train: 1.1733 | Val: 1.1840 | LR: 0.000500
Early stopping at epoch 7


In [ ]:
def calculate_hybrid_metrics_full(model, loader, k=10, threshold=7.0):
    model.eval()
    user_results = defaultdict(list)
    total_mse = 0
    with torch.no_grad():
        for u, i, r in loader:
            u, i, r = u.to(device), i.to(device), r.to(device)
            preds = model(u, i)
            total_mse += nn.functional.mse_loss(preds, r, reduction='sum').item()
            u_np, r_np, p_np = u.cpu().numpy(), r.cpu().numpy(), preds.cpu().numpy()
            for idx in range(len(u_np)):
                user_results[int(u_np[idx])].append((float(p_np[idx]), float(r_np[idx])))

    final_rmse = np.sqrt(total_mse / len(loader.dataset))
    precisions, recalls, ndcgs = [], [], []

    for uid, ratings in user_results.items():
        ratings.sort(key=lambda x: x[0], reverse=True)
        n_rel = sum(1 for p, r in ratings if r >= threshold)
        n_rel_and_rec_k = sum(1 for p, r in ratings[:k] if r >= threshold)
        precisions.append(n_rel_and_rec_k / k)
        recalls.append(n_rel_and_rec_k / n_rel if n_rel > 0 else 0)

        dcg = sum([r / np.log2(idx + 2) for idx, (p, r) in enumerate(ratings[:k])])
        ratings.sort(key=lambda x: x[1], reverse=True)
        idcg = sum([r / np.log2(idx + 2) for idx, (p, r) in enumerate(ratings[:k])])
        if idcg > 0: ndcgs.append(dcg / idcg)

    return {
        "RMSE": final_rmse,
        "Precision@10": np.mean(precisions),
        "Recall@10": np.mean(recalls),
        "NDCG@10": np.mean(ndcgs)
    }

# Load best weights and run test
model.load_state_dict(torch.load(f"{variant_v2_dir}/best_hybrid_v2_total_context.pth"))
v2_test_results = calculate_hybrid_metrics_full(model, test_loader)

print(f"\n--- V2 TOTAL CONTEXT TEST RESULTS ---")
for m, v in v2_test_results.items(): print(f"{m}: {v:.4f}")


--- V2 TOTAL CONTEXT TEST RESULTS ---
RMSE: 1.2428
Precision@10: 0.2673
Recall@10: 0.9060
NDCG@10: 0.9848


In [ ]:
def recommend_for_user(user_name, k=10):
    # Map username to ID
    user_mapping = pl.read_csv(f"{drive_path}/processed_data/csv_backups/user_mapping.csv")
    user_row = user_mapping.filter(pl.col("Username") == user_name)

    if user_row.is_empty():
        return "User not found."

    u_idx = user_row["user_id"][0]

    model.eval()
    # Score all items for this user
    all_i_ids = torch.arange(ACTUAL_NUM_ITEMS).to(device)
    u_ids = torch.full((ACTUAL_NUM_ITEMS,), u_idx, dtype=torch.long).to(device)

    with torch.no_grad():
        scores = model(u_ids, all_i_ids).cpu().numpy()

    # Get Top K
    top_indices = np.argsort(scores)[-k:][::-1]

    # Map back to Game Names
    return items_df.filter(pl.col("item_id").is_in(top_indices)).select(["Name", "AvgRating"])

In [ ]:
print(recommend_for_user("007poptart"))

shape: (10, 2)
┌─────────────────────────────────┬───────────┐
│ Name                            ┆ AvgRating │
│ ---                             ┆ ---       │
│ str                             ┆ f64       │
╞═════════════════════════════════╪═══════════╡
│ 1817                            ┆ 8.68786   │
│ Eclipse: Second Dawn for the G… ┆ 8.67526   │
│ Anachrony: Infinity Box         ┆ 9.07609   │
│ Uprising: Curse of the Last Em… ┆ 8.52968   │
│ Euthia: Torment of Resurrectio… ┆ 8.87023   │
│ Clash of Cultures: Monumental … ┆ 8.62371   │
│ Middara: Unintentional Malum –… ┆ 9.02596   │
│ Ultimate Railroads              ┆ 8.39017   │
│ Great Western Trail (Second Ed… ┆ 8.45802   │
│ Ark Nova                        ┆ 8.47839   │
└─────────────────────────────────┴───────────┘


In [ ]:
print(recommend_for_user("0void0"))

shape: (10, 2)
┌─────────────────────────────────┬───────────┐
│ Name                            ┆ AvgRating │
│ ---                             ┆ ---       │
│ str                             ┆ f64       │
╞═════════════════════════════════╪═══════════╡
│ 1817                            ┆ 8.68786   │
│ Eclipse: Second Dawn for the G… ┆ 8.67526   │
│ Sleeping Gods                   ┆ 8.51042   │
│ Anachrony: Infinity Box         ┆ 9.07609   │
│ Uprising: Curse of the Last Em… ┆ 8.52968   │
│ Euthia: Torment of Resurrectio… ┆ 8.87023   │
│ Clash of Cultures: Monumental … ┆ 8.62371   │
│ Middara: Unintentional Malum –… ┆ 9.02596   │
│ Great Western Trail (Second Ed… ┆ 8.45802   │
│ Ark Nova                        ┆ 8.47839   │
└─────────────────────────────────┴───────────┘
